# 🛡️ AI-Based Explainable & Adversarial-Resistant NIDS
### **Academic Research Prototype (NSL-KDD Dataset)**

This Google Colab Notebook runs the complete end-to-end Network Intrusion Detection System (NIDS) pipeline step-by-step:
1. **Environment Setup & Repo Cloning**
2. **Data Downloading & Preprocessing (SMOTE, Scaling, Encoding)**
3. **Exploratory Data Analysis (EDA)**
4. **Model Training (Random Forest, XGBoost, Keras DNN)**
5. **Model Evaluation & Metric Comparison**
6. **SHAP Explainable AI (XAI)**
7. **Adversarial Robustness Testing (FGSM & PGD Evasion Attacks)**
8. **Adversarial Retraining & Defense**
9. **Final Summary Report**

--- 
## 📍 Step 1: Environment Setup & Package Installation
**What this block does:**
- Clones the project repository from GitHub.
- Installs necessary libraries (`xgboost`, `imbalanced-learn`, `shap`, `adversarial-robustness-toolbox`).
- Sets a deterministic random seed (`42`) across Python, NumPy, and TensorFlow for 100% reproducible results.

In [ ]:
# Clone repository if running in Colab
import os
if not os.path.exists('AI-NIDS'):
    !git clone https://github.com/thekrishagarwal2006/AI-NIDS.git
    %cd AI-NIDS
else:
    %cd AI-NIDS

# Install dependencies
!pip install -q xgboost imbalanced-learn shap adversarial-robustness-toolbox

# Verify setup and set seed
from src.utils import set_seed, ensure_directories
set_seed(42)
ensure_directories()
print("✅ Step 1 complete: Environment configured and repository loaded!")

--- 
## 📍 Step 2: Download & Preprocess NSL-KDD Dataset
**What this block does:**
- Downloads `KDDTrain+.txt` (125,973 flows) and `KDDTest+.txt` (22,544 untouched flows).
- Maps 39 sub-attack types into 4 major threat categories (DoS, Probe, R2L, U2R) + Normal traffic.
- Applies One-Hot Encoding to categorical features (`protocol_type`, `service`, `flag`).
- Normalizes numerical values using `StandardScaler`.
- Applies **SMOTE** (Synthetic Minority Over-sampling Technique) to balance rare attack classes in training data.

In [ ]:
from src.utils import download_nsl_kdd
from src.data_preprocessing import load_and_preprocess_data

# Download dataset files
train_path, test_path = download_nsl_kdd("data")

# Preprocess data
(
    X_train, y_train,
    X_test, y_test,
    df_train_raw, df_test_raw,
    feature_names, label_encoder
) = load_and_preprocess_data(train_path, test_path, use_smote=True)

print(f"✅ Step 2 complete: Training samples = {X_train.shape[0]}, Test samples = {X_test.shape[0]}, Features = {X_train.shape[1]}")

--- 
## 📍 Step 3: Exploratory Data Analysis (EDA)
**What this block does:**
- Generates distribution plots for attack categories in training vs test sets.
- Plots correlation matrices to identify relationships between network traffic features.
- Saves visual plots into `results/eda/`.

In [ ]:
from src.eda import perform_eda

perform_eda(df_train_raw, results_dir="results/eda")
print("✅ Step 3 complete: EDA figures generated and saved under results/eda/")

--- 
## 📍 Step 4: Model Training (RF, XGBoost, DNN)
**What this block does:**
- Trains 3 ML models on balanced training data:
  1. **Random Forest Classifier** (100 estimators, parallel threads)
  2. **XGBoost Classifier** (Gradient Boosted Decision Trees)
  3. **Keras Deep Neural Network (DNN)** (Multi-layer perceptron with BatchNorm and Dropout)
- Saves trained models into `models/`.

In [ ]:
from src.train_models import train_all_models

models_dict = train_all_models(X_train, y_train, X_test, y_test, models_dir="models")
print("✅ Step 4 complete: All 3 models trained and saved to models/")

--- 
## 📍 Step 5: Model Evaluation & Metric Comparison
**What this block does:**
- Evaluates all 3 trained models on untouched test set (`KDDTest+.txt`).
- Calculates Accuracy, Macro Precision, Recall, Macro F1, Weighted F1, and Normal False Positive Rate (FPR).
- Selects the best performing model based on F1-score.

In [ ]:
from src.evaluate_models import evaluate_all_models

best_model_name, best_model, df_results = evaluate_all_models(
    models_dict, X_test, y_test, label_encoder, results_dir="results/models"
)

print(f"\n🏆 Best Model Identified: {best_model_name}")
display(df_results)

--- 
## 📍 Step 6: SHAP Explainable AI (XAI)
**What this block does:**
- Calculates SHAP (SHapley Additive exPlanations) values for model predictions.
- Plots Global Feature Importance (Bar Plot & Summary Beeswarm Plot).
- Computes local attributions (positive & negative contributions) for individual network traffic flows.

In [ ]:
from src.shap_explainability import compute_shap_explanations, explain_single_prediction

explainer, shap_matrix, df_sample = compute_shap_explanations(
    best_model, best_model_name, X_test, feature_names, label_encoder, X_train=X_train, results_dir="results/shap"
)

# Explain single sample prediction
sample_idx = 0
local_explanation, _ = explain_single_prediction(
    best_model, X_test[sample_idx], feature_names, label_encoder, explainer=explainer, sample_idx=sample_idx, results_dir="results/shap"
)
print("\nSample Local SHAP Explanation:")
print(local_explanation)

--- 
## 📍 Step 7: Adversarial Robustness Testing (FGSM & PGD Attacks)
**What this block does:**
- Tests model vulnerability under adversarial evasion attacks using Adversarial Robustness Toolbox (ART).
- Evaluates **Fast Gradient Sign Method (FGSM)** and **Projected Gradient Descent (PGD)** across perturbation values ($\epsilon = 0.05, 0.1, 0.2$).

In [ ]:
from src.adversarial_attacks import evaluate_adversarial_robustness

dnn_model = models_dict['DNN']
df_adv_eval = evaluate_adversarial_robustness(
    dnn_model, X_test, y_test, epsilons=[0.05, 0.1, 0.2], results_dir="results/adversarial"
)
display(df_adv_eval)

--- 
## 📍 Step 8: Adversarial Retraining & Defense Mechanism
**What this block does:**
- Generates adversarial training samples using FGSM perturbation ($\epsilon = 0.1$).
- Retrains the Deep Neural Network on a mixed dataset (50% clean + 50% adversarial).
- Verifies performance recovery against adversarial attacks.

In [ ]:
from src.adversarial_training import perform_adversarial_training

robust_dnn, adv_summary = perform_adversarial_training(
    dnn_model, X_train, y_train, X_test, y_test,
    eps=0.1, epochs=15, results_dir="results/adversarial", models_dir="models"
)
print("\nAdversarial Defense Summary:")
for key, val in adv_summary.items():
    print(f"  {key}: {val}")

--- 
## 📍 Step 9: Generate Final Comprehensive Summary Report
**What this block does:**
- Compiles all quantitative results into `results/final_report.txt`.

In [ ]:
from main import generate_final_report

generate_final_report(best_model_name, df_results, adv_summary, report_path="results/final_report.txt")
print("🎉 Congratulations! The entire NIDS pipeline completed successfully!")